# Live vs. Backtest Reconciliation

Once real trades have accumulated via `daily_runner.py`, this notebook is the real test of
whether the backtest generalizes: compares actual realized live P&L against what the
backtest predicted for the identical period, universe, and config.

In [1]:
# Package is pip-installed editable, no sys.path hacking needed
from momentum_trading.execution.live_signal import measure_live_performance, fetch_live_prices
from momentum_trading.backtest.momentum_backtest import run_custom_backtest, BacktestConfig
from momentum_trading.core.paths import logs_dir
import pandas as pd

## 1. Define the reconciliation window and config

Must match exactly what `daily_runner.py` actually used for this period, pull these from
your `config.yaml`, not from memory, or the comparison isn't apples-to-apples.

In [2]:
# EDIT: match your real config.yaml for the portfolio/period being reconciled
portfolio_name = "portfolio1"
tickers = ["SPY", "QQQ", "XLK", "XLF", "XLE", "XLY", "XLP", "XLU", "GLD", "TLT", "BIL"]
start_date = "2026-01-01"
# end_date is compared against full timestamps (e.g. 2026-07-11T03:29:42), and a bare date
# string parses as that day's midnight, so using today's date here EXCLUDES today's own
# trades. Use tomorrow's date (or later) to make sure today's activity is included.
end_date = "2026-07-12"
initial_capital = 1000.0

cfg = BacktestConfig(
    holding_period=1, initial_capital=initial_capital, commission=0.0,
    drift_threshold=0.03, min_trade_size=25.0,
    use_regime_filter=True, regime_benchmark="SPY", regime_sma_window=150,
    max_position_weight=0.35,
)

## 2. Real live performance (from the actual trade log)

**Prerequisite:** `logs/live_trades_log_<portfolio_name>.csv` must already exist, it's only
written by `daily_runner.py` (dry-run or live, see `docs/RUNNING.md`), never by this notebook.
If you haven't run `daily-runner` for this portfolio yet, the next cell will raise
`FileNotFoundError`, that's expected, not a bug: there's nothing real to reconcile yet.

In [3]:
daily_prices = fetch_live_prices(tickers)
latest_prices = daily_prices.iloc[-1].to_dict()

live_perf = measure_live_performance(
    start_date=start_date, end_date=end_date,
    latest_prices=latest_prices,
    log_path=str(logs_dir() / f"live_trades_log_{portfolio_name}.csv"),
    initial_capital=initial_capital,
)
print(f"Live realized P&L:   ${live_perf['realized_pnl']:,.2f}")
print(f"Live unrealized P&L: ${live_perf['unrealized_pnl']:,.2f}")
print(f"Live total P&L:      ${live_perf['total_pnl']:,.2f}")
if 'total_return_pct' in live_perf:
    print(f"Live total return:   {live_perf['total_return_pct']:.2%}")

2026-07-18 10:47:14,057 | INFO | Fetching live prices for 11 tickers, 2025-06-13 to 2026-07-18


Determining best data source using SPY...
Data retrieved from yf for SPY
Using yf for all tickers
Data retrieved from yf for SPY
Data retrieved from yf for QQQ
Data retrieved from yf for XLK
Data retrieved from yf for XLF
Data retrieved from yf for XLE
Data retrieved from yf for XLY
Data retrieved from yf for XLP
Data retrieved from yf for XLU
Data retrieved from yf for GLD
Data retrieved from yf for TLT
Data retrieved from yf for BIL
Live realized P&L:   $0.00
Live unrealized P&L: $-16.73
Live total P&L:      $-16.73
Live total return:   -1.67%


## 3. Backtested performance over the SAME period, universe, and config

In [4]:
# Rebuild the same signal used live, restricted to the reconciliation window
monthly_prices = daily_prices.resample("ME").last()
scores = monthly_prices.ffill().pct_change(periods=12).dropna(how="all")
ranks = scores.rank(axis=1, ascending=False)
top_etfs_monthly = ranks.apply(lambda x: x.nsmallest(10).index.tolist(), axis=1)

picks_window = top_etfs_monthly[(top_etfs_monthly.index >= start_date) & (top_etfs_monthly.index <= end_date)]
prices_window = daily_prices[(daily_prices.index >= start_date) & (daily_prices.index <= end_date)]

backtest_df = run_custom_backtest(picks_window, prices_window, **cfg.__dict__)
backtest_pnl = backtest_df["Month End Portfolio Value"].iloc[-1] - initial_capital if len(backtest_df) else float("nan")
print(f"Backtested P&L over the same window: ${backtest_pnl:,.2f}")
backtest_df.attrs.get("tearsheet")

2026-07-18 10:47:22,358 | INFO | Backtest period: 2026-06-30 to 2026-07-10 | 1 rebalance dates.
2026-07-18 10:47:22,372 | INFO | Backtest complete. Total commission: $0.00 | Total slippage cost (est.): $0.00
2026-07-18 10:47:22,407 | INFO | ============================================================
2026-07-18 10:47:22,408 | INFO | TEARSHEET
2026-07-18 10:47:22,410 | INFO | CAGR:                 0.00%
2026-07-18 10:47:22,413 | INFO | Annualized Vol:       nan%
2026-07-18 10:47:22,415 | INFO | Sharpe Ratio:         nan
2026-07-18 10:47:22,417 | INFO | Sortino Ratio:        nan
2026-07-18 10:47:22,418 | INFO | Max Drawdown:         0.00%
2026-07-18 10:47:22,420 | INFO | Calmar Ratio:         nan
2026-07-18 10:47:22,424 | INFO | Monthly Win Rate:     0.0%
2026-07-18 10:47:22,425 | INFO | Beta vs SPY:          nan
2026-07-18 10:47:22,427 | INFO | Annualized Alpha:     nan%
2026-07-18 10:47:22,434 | INFO | Total Commission:     $0.00
2026-07-18 10:47:22,437 | INFO | Total Est. Slippage:  $

Backtested P&L over the same window: $0.00


{'CAGR': np.float64(0.0),
 'AnnVol': np.float64(nan),
 'Sharpe': nan,
 'Sortino': nan,
 'MaxDrawdown': np.float64(0.0),
 'Calmar': nan,
 'WinRate': np.float64(0.0),
 'Beta': nan,
 'Alpha': nan,
 'TotalCommission': 0.0,
 'TotalSlippage': 0.0,
 'TotalTurnover': 0}

## 4. Side-by-side comparison and divergence flag

In [5]:
comparison = pd.DataFrame({
    "Live": [live_perf["total_pnl"]],
    "Backtest": [backtest_pnl],
}, index=["Total P&L"])
comparison["Difference"] = comparison["Live"] - comparison["Backtest"]
comparison["Difference %"] = comparison["Difference"] / comparison["Backtest"].abs()
display(comparison)

DIVERGENCE_THRESHOLD = 0.25  # flag if live diverges from backtest by more than 25%
diff_pct = abs(comparison["Difference %"].iloc[0])
if diff_pct > DIVERGENCE_THRESHOLD:
    print(f"\n\u26a0\ufe0f  MATERIAL DIVERGENCE: live P&L differs from backtest by {diff_pct:.1%}.")
    print("Investigate: slippage model accuracy, data vendor discrepancies, execution timing,")
    print("or whether daily_runner.py's actual config matches what was backtested here.")
else:
    print(f"\nLive and backtest are within {diff_pct:.1%} of each other, no major reconciliation concern.")

,Live,Backtest,Difference,Difference %
Total P&L,-16.726896,0.0,-16.726896,-inf



⚠️  MATERIAL DIVERGENCE: live P&L differs from backtest by inf%.
Investigate: slippage model accuracy, data vendor discrepancies, execution timing,
or whether daily_runner.py's actual config matches what was backtested here.
